In [30]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

transactions = [
    ["bread", "milk", "butter"],
    ["bread", "milk"],
    ["bread", "eggs"],
    ["milk", "eggs"],
    ["bread", "milk", "eggs"],
    ["bread", "butter"],
    ["milk", "butter"],
    ["bread", "milk", "butter", "eggs"],
    ["milk", "eggs", "butter"],
    ["bread", "milk", "eggs"],
    ["bread", "butter"],
    ["milk", "butter"],
    ["bread", "milk"],
    ["bread", "eggs", "butter"],
    ["milk", "eggs"],
]

len(transactions)

15

In [31]:
te = TransactionEncoder()
encoded_data = te.fit_transform(transactions)

basket = pd.DataFrame(
    encoded_data,
    columns = te.columns_
)

frequent_itemsets = apriori(
    basket,
    min_support = 0.2,
    use_colnames = True
)
frequent_itemsets

,support,itemsets
0,0.666667,frozenset({bread})
1,0.533333,frozenset({butter})
2,0.533333,frozenset({eggs})
3,0.733333,frozenset({milk})
4,0.333333,"frozenset({butter, bread})"
5,0.333333,"frozenset({eggs, bread})"
6,0.400000,"frozenset({milk, bread})"
7,0.200000,"frozenset({eggs, butter})"
8,0.333333,"frozenset({milk, butter})"
9,0.400000,"frozenset({milk, eggs})"


In [21]:
rules = association_rules(
    frequent_itemsets,
    metric = "confidence",
    min_threshold = 0.5
)

rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]]

,antecedents,consequents,support,confidence,lift
0,frozenset({butter}),frozenset({bread}),0.333333,0.625000,0.937500
1,frozenset({bread}),frozenset({butter}),0.333333,0.500000,0.937500
2,frozenset({eggs}),frozenset({bread}),0.333333,0.625000,0.937500
3,frozenset({bread}),frozenset({eggs}),0.333333,0.500000,0.937500
4,frozenset({milk}),frozenset({bread}),0.400000,0.545455,0.818182
5,frozenset({bread}),frozenset({milk}),0.400000,0.600000,0.818182
6,frozenset({butter}),frozenset({milk}),0.333333,0.625000,0.852273
7,frozenset({eggs}),frozenset({milk}),0.400000,0.750000,1.022727
8,frozenset({milk}),frozenset({eggs}),0.400000,0.545455,1.022727
9,"frozenset({milk, eggs})",frozenset({bread}),0.200000,0.500000,0.750000


In [29]:
import mlflow
from pathlib import Path

project_dir = Path.cwd().parent
db_path = project_dir / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{db_path.as_posix()}")
mlflow.set_experiment("Market Basket Association Rules")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment created successfully")

2026/10/17 07:47:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/17 07:47:34 INFO mlflow.store.db.utils: Updating database tables
2026/10/17 07:47:37 INFO mlflow.tracking.fluent: Experiment with name 'Market Basket Association Rules' does not exist. Creating a new experiment.


Tracking URI: sqlite:///c:/Users/Mumba Mumba/Desktop/association-rule-mlops/mlflow.db
Experiment created successfully


In [33]:
with mlflow.start_run(run_name="apriori_baseline"):

    min_support = 0.2
    min_confidence = 0.5

    itemsets = apriori(
        basket,
        min_support=min_support,
        use_colnames=True
    )

    rules = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=min_confidence
    )

    mlflow.log_param("algorithm", "Apriori")
    mlflow.log_param("min_support", min_support)
    mlflow.log_param("min_confidence", min_confidence)

    mlflow.log_metric("number_of_itemsets", len(itemsets))
    mlflow.log_metric("number_of_rules", len(rules))

    if not rules.empty:
        mlflow.log_metric("average_lift", rules["lift"].mean())

    rules.to_csv("rules_baseline.csv", index=False)
    mlflow.log_artifact("rules_baseline.csv")

    print("Baseline itemsets:", len(itemsets))
    print("Baseline rules:", len(rules))

Baseline itemsets: 11
Baseline rules: 12


In [35]:
with mlflow.start_run(run_name="apriori_stricter"):

    min_support = 0.3
    min_confidence = 0.7

    itemsets_2 = apriori(
        basket,
        min_support=min_support,
        use_colnames=True
    )

    rules_2 = association_rules(
        itemsets_2,
        metric="confidence",
        min_threshold=min_confidence
    )

    mlflow.log_param("algorithm", "Apriori")
    mlflow.log_param("min_support", min_support)
    mlflow.log_param("min_confidence", min_confidence)

    mlflow.log_metric("number_of_itemsets", len(itemsets_2))
    mlflow.log_metric("number_of_rules", len(rules_2))

    if not rules_2.empty:
        mlflow.log_metric("average_lift", rules_2["lift"].mean())

    rules_2.to_csv("rules_stricter.csv", index=False)
    mlflow.log_artifact("rules_stricter.csv")

    print("Stricter itemsets:", len(itemsets_2))
    print("Stricter rules:", len(rules_2))

Stricter itemsets: 9
Stricter rules: 1
